<a href="https://colab.research.google.com/github/littlestuart07/Online-Retail-Analytics-Project/blob/main/Digital%20Assignment%203/DA3_Final_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install plotly scipy statsmodels xgboost lightgbm --quiet
print("done")

done


In [24]:
import numpy as np
import pandas as pd
import sqlite3, json, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier

from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

df = pd.read_csv("customer_features.csv")
print("Customer records:", df.shape)
df.head()

Customer records: (5281, 13)


,Customer ID,Recency,Frequency,Monetary,AvgOrderValue,AvgQuantity,TotalQuantity,OutlierOrders,Country,Churn,HighValue,Segment,CountryGroup
0,12346,235.117361,12,77556.46,6463.038333,2184.852941,74285,1,United Kingdom,1,1,2,United Kingdom
1,12347,39.168056,6,3402.39,567.065000,12.798780,2099,6,Iceland,0,1,3,Other
2,12348,158.085417,4,1709.40,427.350000,52.020833,2497,19,Finland,0,0,3,Other
3,12349,317.185417,3,2671.14,890.380000,9.735294,993,1,Italy,0,1,1,Other
4,12350,219.867361,1,334.40,334.400000,11.588235,197,0,Norway,1,0,3,Other


In [25]:
feature_cols = ["Recency", "Frequency", "Monetary", "AvgOrderValue",
                "AvgQuantity", "TotalQuantity", "OutlierOrders", "CountryGroup"]
df_model = pd.get_dummies(df[feature_cols + ["Churn"]], columns=["CountryGroup"], drop_first=True)

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
X_all_s   = StandardScaler().fit_transform(X)

models = {
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(32,16), max_iter=500, random_state=42),
    "Gradient Boosting":    GradientBoostingClassifier(random_state=42),
    "Logistic Regression":  LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost":              XGBClassifier(eval_metric="logloss", random_state=42),
}

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Features: {X.shape[1]}")

Train: 3960 | Test: 1321 | Features: 12


In [26]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = {}

for name, m in models.items():
    scores = cross_val_score(m, X_all_s, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    fold_scores[name] = scores
    print(f"{name:24s} {np.round(scores,4)}  mean={scores.mean():.4f}")

cv_folds_df = pd.DataFrame(fold_scores)
cv_folds_df.index = [f"Fold {i+1}" for i in range(5)]
cv_folds_df.to_csv("da3_cv_folds.csv")
cv_folds_df.round(4)

Neural Network (MLP)     [0.7967 0.8042 0.8    0.7993 0.7817]  mean=0.7964
Gradient Boosting        [0.7993 0.8006 0.798  0.8024 0.7799]  mean=0.7960
Logistic Regression      [0.792  0.7938 0.789  0.7975 0.7671]  mean=0.7879
Random Forest            [0.7849 0.785  0.7883 0.7855 0.768 ]  mean=0.7823
XGBoost                  [0.7509 0.7626 0.7701 0.77   0.7494]  mean=0.7606


,Neural Network (MLP),Gradient Boosting,Logistic Regression,Random Forest,XGBoost
Fold 1,0.7967,0.7993,0.7920,0.7849,0.7509
Fold 2,0.8042,0.8006,0.7938,0.7850,0.7626
Fold 3,0.8000,0.7980,0.7890,0.7883,0.7701
Fold 4,0.7993,0.8024,0.7975,0.7855,0.7700
Fold 5,0.7817,0.7799,0.7671,0.7680,0.7494


In [27]:
rows = []
for name, s in fold_scores.items():
    mean, sd, n = s.mean(), s.std(ddof=1), len(s)
    tcrit = stats.t.ppf(0.975, df=n-1)
    margin = tcrit * sd / np.sqrt(n)
    rows.append({"Model": name, "Mean_AUC": mean, "Std": sd,
                 "CI_Lower_95": mean - margin, "CI_Upper_95": mean + margin})

ci_df = pd.DataFrame(rows).sort_values("Mean_AUC", ascending=False).reset_index(drop=True)
ci_df.to_csv("da3_confidence_intervals.csv", index=False)
print("95% confidence intervals for mean ROC-AUC (5 folds)")
ci_df.round(4)

95% confidence intervals for mean ROC-AUC (5 folds)


,Model,Mean_AUC,Std,CI_Lower_95,CI_Upper_95
0,Neural Network (MLP),0.7964,0.0086,0.7856,0.8071
1,Gradient Boosting,0.7960,0.0091,0.7847,0.8074
2,Logistic Regression,0.7879,0.0120,0.7729,0.8028
3,Random Forest,0.7823,0.0081,0.7722,0.7924
4,XGBoost,0.7606,0.0100,0.7481,0.7731


In [28]:
names = list(fold_scores.keys())
results = []

for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = fold_scores[names[i]], fold_scores[names[j]]
        t_stat, t_p = stats.ttest_rel(a, b)
        try:
            w_stat, w_p = stats.wilcoxon(a, b)
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        results.append({
            "Model A": names[i], "Model B": names[j],
            "Mean Diff": a.mean() - b.mean(),
            "t-statistic": t_stat, "t p-value": t_p,
            "Wilcoxon p": w_p,
            "Significant (5%)": "Yes" if t_p < 0.05 else "No"
        })

pairwise_df = pd.DataFrame(results)
pairwise_df.to_csv("da3_pairwise_tests.csv", index=False)
print("Paired t-test and Wilcoxon signed-rank test on 5-fold ROC-AUC")
pairwise_df.round(4)

Paired t-test and Wilcoxon signed-rank test on 5-fold ROC-AUC


,Model A,Model B,Mean Diff,t-statistic,t p-value,Wilcoxon p,Significant (5%)
0,Neural Network (MLP),Gradient Boosting,0.0003,0.2444,0.8190,1.0000,No
1,Neural Network (MLP),Logistic Regression,0.0085,3.6655,0.0215,0.0625,Yes
2,Neural Network (MLP),Random Forest,0.0141,10.2558,0.0005,0.0625,Yes
3,Neural Network (MLP),XGBoost,0.0358,10.6909,0.0004,0.0625,Yes
4,Gradient Boosting,Logistic Regression,0.0082,6.0590,0.0037,0.0625,Yes
5,Gradient Boosting,Random Forest,0.0137,10.7181,0.0004,0.0625,Yes
6,Gradient Boosting,XGBoost,0.0355,9.7741,0.0006,0.0625,Yes
7,Logistic Regression,Random Forest,0.0056,2.2752,0.0852,0.1875,No
8,Logistic Regression,XGBoost,0.0273,6.3706,0.0031,0.0625,Yes
9,Random Forest,XGBoost,0.0217,6.6798,0.0026,0.0625,Yes


In [29]:
preds = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    preds[name] = m.predict(X_test_s)

yt = y_test.values
mc_rows = []
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a = (preds[names[i]] == yt).astype(int)
        b = (preds[names[j]] == yt).astype(int)
        n01 = int(((a==1)&(b==0)).sum())      # only A correct
        n10 = int(((a==0)&(b==1)).sum())      # only B correct
        tbl = [[int(((a==1)&(b==1)).sum()), n01],
               [n10, int(((a==0)&(b==0)).sum())]]
        res = mcnemar(tbl, exact=False, correction=True)
        mc_rows.append({"Model A": names[i], "Model B": names[j],
                        "A only correct": n01, "B only correct": n10,
                        "Statistic": res.statistic, "p-value": res.pvalue,
                        "Significant (5%)": "Yes" if res.pvalue < 0.05 else "No"})

mcnemar_df = pd.DataFrame(mc_rows)
mcnemar_df.to_csv("da3_mcnemar_matrix.csv", index=False)
print("Pairwise McNemar tests on the 1,321 test customers")
mcnemar_df.round(4)

Pairwise McNemar tests on the 1,321 test customers


,Model A,Model B,A only correct,B only correct,Statistic,p-value,Significant (5%)
0,Neural Network (MLP),Gradient Boosting,58,37,4.2105,0.0402,Yes
1,Neural Network (MLP),Logistic Regression,57,47,0.7788,0.3775,No
2,Neural Network (MLP),Random Forest,92,63,5.0581,0.0245,Yes
3,Neural Network (MLP),XGBoost,137,86,11.2108,0.0008,Yes
4,Gradient Boosting,Logistic Regression,54,65,0.8403,0.3593,No
5,Gradient Boosting,Random Forest,76,68,0.3403,0.5597,No
6,Gradient Boosting,XGBoost,112,82,4.3351,0.0373,Yes
7,Logistic Regression,Random Forest,103,84,1.7326,0.1881,No
8,Logistic Regression,XGBoost,142,101,6.5844,0.0103,Yes
9,Random Forest,XGBoost,97,75,2.5640,0.1093,No


In [30]:
probas = {name: m.predict_proba(X_test_s)[:,1] for name, m in models.items()}
rng = np.random.default_rng(42)
n = len(yt)

boot_rows = []
for name, p in probas.items():
    aucs = []
    for _ in range(1000):
        idx = rng.integers(0, n, n)
        if len(np.unique(yt[idx])) < 2:
            continue
        aucs.append(roc_auc_score(yt[idx], p[idx]))
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    boot_rows.append({"Model": name, "Test_AUC": roc_auc_score(yt, p),
                      "Boot_CI_Lower": lo, "Boot_CI_Upper": hi})

boot_df = pd.DataFrame(boot_rows).sort_values("Test_AUC", ascending=False).reset_index(drop=True)
boot_df.to_csv("da3_bootstrap_ci.csv", index=False)
print("Bootstrap 95% CI for test ROC-AUC (1,000 resamples)")
boot_df.round(4)

Bootstrap 95% CI for test ROC-AUC (1,000 resamples)


,Model,Test_AUC,Boot_CI_Lower,Boot_CI_Upper
0,Neural Network (MLP),0.8070,0.7835,0.8273
1,Gradient Boosting,0.8054,0.7843,0.8283
2,Logistic Regression,0.7973,0.7721,0.8207
3,Random Forest,0.7872,0.7644,0.8105
4,XGBoost,0.7722,0.7449,0.7973


In [31]:
final_rows = []
for name, m in models.items():
    pred, proba = preds[name], probas[name]
    final_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(yt, pred),
        "Precision": precision_score(yt, pred),
        "Recall": recall_score(yt, pred),
        "F1": f1_score(yt, pred),
        "Test ROC-AUC": roc_auc_score(yt, proba),
        "CV Mean AUC": fold_scores[name].mean(),
        "CV Std": fold_scores[name].std(ddof=1),
    })

final_df = pd.DataFrame(final_rows).sort_values("Test ROC-AUC", ascending=False).reset_index(drop=True)
final_df.to_csv("da3_final_comparison.csv", index=False)
print("Final consolidated comparison")
final_df.round(4)

Final consolidated comparison


,Model,Accuracy,Precision,Recall,F1,Test ROC-AUC,CV Mean AUC,CV Std
0,Neural Network (MLP),0.7441,0.7595,0.8021,0.7802,0.8070,0.7964,0.0086
1,Gradient Boosting,0.7282,0.7369,0.8088,0.7712,0.8054,0.7960,0.0091
2,Logistic Regression,0.7366,0.7597,0.7821,0.7708,0.7973,0.7879,0.0120
3,Random Forest,0.7222,0.7378,0.7901,0.7631,0.7872,0.7823,0.0081
4,XGBoost,0.7055,0.7247,0.7741,0.7485,0.7722,0.7606,0.0100


In [32]:
# Gradient Boosting with the tuned threshold from DA2
gb = models["Gradient Boosting"]
all_proba = gb.predict_proba(scaler.transform(X))[:, 1]

scores = pd.DataFrame({
    "customer_id": df["Customer ID"].astype(int),
    "segment": df["Segment"].astype(int),
    "monetary": df["Monetary"].round(2),
    "frequency": df["Frequency"].astype(int),
    "recency": df["Recency"].round(1),
    "high_value": df["HighValue"].astype(int),
    "churn_actual": df["Churn"].astype(int),
    "churn_probability": all_proba.round(4),
})
scores["churn_predicted"] = (scores["churn_probability"] >= 0.453).astype(int)
scores["risk_band"] = pd.cut(scores["churn_probability"], bins=[0, 0.3, 0.6, 1.0],
                             labels=["Low", "Medium", "High"]).astype(str)
scores["value_tier"] = pd.qcut(scores["monetary"], q=4,
                               labels=["Bronze", "Silver", "Gold", "Platinum"]).astype(str)

# revenue exposed per customer = how much they are worth x how likely they are to leave
scores["revenue_at_risk"] = (scores["monetary"] * scores["churn_probability"]).round(2)

def action(r):
    if r.risk_band == "High":
        return "Urgent retention call" if r.value_tier in ("Gold","Platinum") else "Automated win-back offer"
    if r.risk_band == "Medium":
        return "Proactive engagement" if r.value_tier in ("Gold","Platinum") else "Monitor"
    return "Loyalty / upsell" if r.value_tier in ("Gold","Platinum") else "Maintain"

scores["recommended_action"] = scores.apply(action, axis=1)
scores = scores.sort_values("revenue_at_risk", ascending=False).reset_index(drop=True)
scores.to_csv("da3_customer_scores.csv", index=False)

print("Customers scored:", len(scores))
print(scores["recommended_action"].value_counts(), "\n")
print("Total revenue at risk: GBP", f"{scores['revenue_at_risk'].sum():,.0f}")
scores.head(10)

Customers scored: 5281
recommended_action
Automated win-back offer    2046
Proactive engagement        1091
Loyalty / upsell             980
Monitor                      586
Urgent retention call        569
Maintain                       9
Name: count, dtype: int64 

Total revenue at risk: GBP 3,668,134


,customer_id,segment,monetary,frequency,recency,high_value,churn_actual,churn_probability,churn_predicted,risk_band,value_tier,revenue_at_risk,recommended_action
0,12346,2,77556.46,12,235.1,1,1,0.9023,1,High,Platinum,69979.19,Urgent retention call
1,16754,2,65500.07,29,281.8,1,1,0.8149,1,High,Platinum,53376.01,Urgent retention call
2,15749,2,44534.30,3,145.0,1,1,0.8328,1,High,Platinum,37088.17,Urgent retention call
3,18102,4,456780.49,112,8.0,1,0,0.0770,0,Low,Platinum,35172.10,Loyalty / upsell
4,14646,4,426699.12,120,2.1,1,0,0.0812,0,Low,Platinum,34647.97,Loyalty / upsell
5,17850,4,51208.87,155,281.9,1,1,0.6654,1,High,Platinum,34074.38,Urgent retention call
6,15098,3,39916.50,3,91.9,1,1,0.7863,1,High,Platinum,31386.34,Urgent retention call
7,13902,1,34095.26,5,542.0,1,1,0.9091,1,High,Platinum,30996.00,Urgent retention call
8,13093,2,54144.25,55,185.0,1,1,0.5667,1,Medium,Platinum,30683.55,Proactive engagement
9,14156,4,294916.33,143,2.9,1,0,0.0759,0,Low,Platinum,22384.15,Loyalty / upsell


In [33]:
conn = sqlite3.connect("da2_retail_normalized.sqlite")
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS customer_scores")
cur.execute("""
CREATE TABLE customer_scores (
  score_id          INTEGER PRIMARY KEY AUTOINCREMENT,
  customer_id       INTEGER NOT NULL,
  segment           INTEGER,
  monetary          REAL,
  frequency         INTEGER,
  recency           REAL,
  high_value        INTEGER CHECK (high_value IN (0,1)),
  churn_actual      INTEGER CHECK (churn_actual IN (0,1)),
  churn_probability REAL,
  churn_predicted   INTEGER CHECK (churn_predicted IN (0,1)),
  risk_band         TEXT,
  value_tier        TEXT,
  revenue_at_risk   REAL,
  recommended_action TEXT,
  FOREIGN KEY (customer_id) REFERENCES customers(customer_id))""")
conn.commit()

scores.to_sql("customer_scores", conn, if_exists="append", index=False)
cur.execute("CREATE INDEX idx_scores_customer ON customer_scores(customer_id)")
cur.execute("CREATE INDEX idx_scores_action   ON customer_scores(recommended_action)")
conn.commit()

print(pd.read_sql("""
SELECT recommended_action, COUNT(*) AS customers,
       ROUND(AVG(churn_probability),3) AS avg_risk,
       ROUND(SUM(monetary),2)          AS lifetime_value,
       ROUND(SUM(revenue_at_risk),2)   AS revenue_at_risk
FROM customer_scores GROUP BY recommended_action
ORDER BY revenue_at_risk DESC""", conn).to_string(index=False))

      recommended_action  customers  avg_risk  lifetime_value  revenue_at_risk
   Urgent retention call        569     0.733      1383015.30       1061677.31
    Proactive engagement       1091     0.449      2256525.77        996157.84
        Loyalty / upsell        980     0.153      9368647.89        975470.17
Automated win-back offer       2046     0.802       636537.07        494450.50
                 Monitor        586     0.494       282776.80        139625.41
                Maintain          9     0.252         3053.17           752.84


In [34]:
conn = sqlite3.connect("da2_retail_normalized.sqlite")
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS customer_scores")
cur.execute("""
CREATE TABLE customer_scores (
  score_id          INTEGER PRIMARY KEY AUTOINCREMENT,
  customer_id       INTEGER NOT NULL,
  segment           INTEGER,
  monetary          REAL,
  frequency         INTEGER,
  recency           REAL,
  high_value        INTEGER CHECK (high_value IN (0,1)),
  churn_actual      INTEGER CHECK (churn_actual IN (0,1)),
  churn_probability REAL,
  churn_predicted   INTEGER CHECK (churn_predicted IN (0,1)),
  risk_band         TEXT,
  value_tier        TEXT,
  revenue_at_risk   REAL,
  recommended_action TEXT,
  FOREIGN KEY (customer_id) REFERENCES customers(customer_id))""")
conn.commit()

scores.to_sql("customer_scores", conn, if_exists="append", index=False)
cur.execute("CREATE INDEX idx_scores_customer ON customer_scores(customer_id)")
cur.execute("CREATE INDEX idx_scores_action   ON customer_scores(recommended_action)")
conn.commit()

print(pd.read_sql("""
SELECT recommended_action, COUNT(*) AS customers,
       ROUND(AVG(churn_probability),3) AS avg_risk,
       ROUND(SUM(monetary),2)          AS lifetime_value,
       ROUND(SUM(revenue_at_risk),2)   AS revenue_at_risk
FROM customer_scores GROUP BY recommended_action
ORDER BY revenue_at_risk DESC""", conn).to_string(index=False))

      recommended_action  customers  avg_risk  lifetime_value  revenue_at_risk
   Urgent retention call        569     0.733      1383015.30       1061677.31
    Proactive engagement       1091     0.449      2256525.77        996157.84
        Loyalty / upsell        980     0.153      9368647.89        975470.17
Automated win-back offer       2046     0.802       636537.07        494450.50
                 Monitor        586     0.494       282776.80        139625.41
                Maintain          9     0.252         3053.17           752.84


In [35]:
import shutil, os
from google.colab import files

os.makedirs("da3_outputs", exist_ok=True)
for f in os.listdir():
    if f.startswith("da3_") and f.endswith(".csv"):
        shutil.copy(f, "da3_outputs/")
shutil.copy("da2_retail_normalized.sqlite", "da3_outputs/")
shutil.make_archive("DA3_outputs", "zip", "da3_outputs")
files.download("DA3_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
import pandas as pd, numpy as np, plotly.graph_objects as go, plotly.express as px
from plotly.offline import get_plotlyjs

# ---------- load exports ----------
final_df   = pd.read_csv("da3_final_comparison.csv")
folds      = pd.read_csv("da3_cv_folds.csv", index_col=0)
boot       = pd.read_csv("da3_bootstrap_ci.csv")
pairwise   = pd.read_csv("da3_pairwise_tests.csv")
mcn        = pd.read_csv("da3_mcnemar_matrix.csv")
roc_pts    = pd.read_csv("da3_roc_points.csv")
scores     = pd.read_csv("da3_customer_scores.csv")
segprof    = pd.read_csv("da3_segment_profiles.csv")
monthly    = pd.read_csv("da3_monthly_revenue.csv")
feats      = pd.read_csv("customer_features.csv")

# ---------- palette (validated for colour-blind separation) ----------
CAT   = ["#2A6FB0", "#C65A1E", "#0E8F86", "#8E4EA8", "#B8902A"]
RISK  = {"Low": "#0E8F86", "Medium": "#B8902A", "High": "#C65A1E"}
SURF, INK, MUTED, GRID = "#fcfcfb", "#33312e", "#6f6a62", "#eceae4"

def style(fig, h=430, title=""):
    fig.update_layout(template="simple_white", height=h,
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        margin=dict(l=70, r=30, t=55, b=55),
        font=dict(family="Segoe UI, Inter, Arial", size=13, color=INK),
        paper_bgcolor=SURF, plot_bgcolor=SURF,
        legend=dict(orientation="h", y=-0.18, x=0, font=dict(color=MUTED)))
    fig.update_xaxes(showgrid=False, linecolor="#d9d4cb", ticks="outside", tickcolor="#d9d4cb")
    fig.update_yaxes(gridcolor=GRID, linecolor="#d9d4cb", zeroline=False)
    return fig

def table(dfx, h=330, title="", widths=None):
    fig = go.Figure(go.Table(
        columnwidth=widths,
        header=dict(values=[f"<b>{c}</b>" for c in dfx.columns], fill_color="#efece5",
                    align="left", font=dict(color=INK, size=12), height=30),
        cells=dict(values=[dfx[c] for c in dfx.columns], fill_color=SURF, align="left",
                   font=dict(color=INK, size=12), height=26,
                   line=dict(color=GRID, width=1))))
    return style(fig, h, title)

# ================= TAB 1 — OVERVIEW =================
act = (scores.groupby("recommended_action")
       .agg(customers=("customer_id","size"), revenue_at_risk=("revenue_at_risk","sum"))
       .reset_index().sort_values("revenue_at_risk"))

f_act = go.Figure(go.Bar(
    x=act["revenue_at_risk"], y=act["recommended_action"], orientation="h",
    marker=dict(color=CAT[0], line=dict(width=0)),
    text=[f"£{v:,.0f}  ({c:,} customers)" for v, c in zip(act.revenue_at_risk, act.customers)],
    textposition="outside", textfont=dict(color=MUTED, size=12),
    hovertemplate="%{y}<br>Revenue at risk £%{x:,.0f}<extra></extra>"))
f_act.update_xaxes(title="Revenue at risk (GBP)", range=[0, act.revenue_at_risk.max()*1.45])
f_act = style(f_act, 430, "Revenue exposed by recommended action")

risk = (scores.groupby("risk_band")
        .agg(customers=("customer_id","size"), lifetime=("monetary","sum"),
             at_risk=("revenue_at_risk","sum"), avg_prob=("churn_probability","mean"))
        .reindex(["Low","Medium","High"]).reset_index())

f_risk = go.Figure(go.Bar(
    x=risk["risk_band"], y=risk["customers"],
    marker=dict(color=[RISK[b] for b in risk["risk_band"]]),
    text=[f"{v:,}" for v in risk["customers"]], textposition="outside",
    textfont=dict(color=MUTED),
    hovertemplate="%{x} risk<br>%{y:,} customers<extra></extra>"))
f_risk.update_yaxes(title="Customers")
f_risk = style(f_risk, 380, "Customers by churn risk band")

f_riskrev = go.Figure(go.Bar(
    x=risk["risk_band"], y=risk["at_risk"],
    marker=dict(color=[RISK[b] for b in risk["risk_band"]]),
    text=[f"£{v:,.0f}" for v in risk["at_risk"]], textposition="outside",
    textfont=dict(color=MUTED),
    hovertemplate="%{x} risk<br>£%{y:,.0f} at risk<extra></extra>"))
f_riskrev.update_yaxes(title="Revenue at risk (GBP)")
f_riskrev = style(f_riskrev, 380, "Revenue at risk by band")

# ================= TAB 2 — MODELS =================
b = boot.sort_values("Test_AUC")
f_auc = go.Figure(go.Bar(
    x=b["Test_AUC"], y=b["Model"], orientation="h",
    marker=dict(color=CAT[0]),
    error_x=dict(type="data", symmetric=False,
                 array=b["Boot_CI_Upper"]-b["Test_AUC"],
                 arrayminus=b["Test_AUC"]-b["Boot_CI_Lower"],
                 color=MUTED, thickness=1.5, width=5),
    text=[f"{v:.3f}" for v in b["Test_AUC"]], textposition="outside",
    textfont=dict(color=MUTED),
    hovertemplate="%{y}<br>AUC %{x:.4f}<extra></extra>"))
f_auc.update_xaxes(title="Test ROC-AUC (bars show bootstrap 95% CI)", range=[0.70, 0.86])
f_auc = style(f_auc, 420, "Model comparison with confidence intervals")

f_box = go.Figure()
for i, m in enumerate(folds.columns):
    f_box.add_trace(go.Box(y=folds[m], name=m, marker_color=CAT[i % 5],
                           boxpoints="all", jitter=0.4, pointpos=0, marker=dict(size=8),
                           line=dict(width=2)))
f_box.update_yaxes(title="ROC-AUC")
f_box = style(f_box, 430, "Cross-validation score distribution (5 folds)")
f_box.update_layout(showlegend=False)

f_roc = go.Figure()
for i, m in enumerate(roc_pts["Model"].unique()):
    d = roc_pts[roc_pts.Model == m]
    f_roc.add_trace(go.Scatter(x=d.FPR, y=d.TPR, mode="lines", name=m,
                               line=dict(color=CAT[i % 5], width=2),
                               hovertemplate=m+"<br>FPR %{x:.2f} · TPR %{y:.2f}<extra></extra>"))
f_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", name="Random baseline",
                           line=dict(color=MUTED, width=1, dash="dash")))
f_roc.update_xaxes(title="False positive rate"); f_roc.update_yaxes(title="True positive rate")
f_roc = style(f_roc, 480, "ROC curves on the test set")

t_final = table(final_df.round(4), 300, "Full metric table")
t_pair  = table(pairwise.round(4), 380, "Paired t-test and Wilcoxon across folds")
t_mcn   = table(mcn.round(4), 380, "Pairwise McNemar tests on the test set")

# ================= TAB 3 — SEGMENTS =================
seg_df = feats.copy()
seg_df["Segment"] = "Segment " + seg_df["Segment"].astype(str)
f_seg = px.scatter(seg_df, x="Recency", y="Monetary", color="Segment", size="Frequency",
                   size_max=26, log_y=True, opacity=0.65,
                   color_discrete_sequence=CAT,
                   hover_data={"Frequency": True, "Monetary": ":,.0f", "Recency": ":.0f"})
f_seg.update_traces(marker=dict(line=dict(width=1, color=SURF)))
f_seg.update_xaxes(title="Recency (days before cutoff)")
f_seg.update_yaxes(title="Monetary (GBP, log scale)")
f_seg = style(f_seg, 520, "Customer segments: recency against spend")

sp = segprof.copy()
f_segchurn = go.Figure(go.Bar(
    x="Segment " + sp["Segment"].astype(str), y=sp["ChurnRate"],
    marker=dict(color=CAT[:len(sp)]),
    text=[f"{v:.1f}%" for v in sp["ChurnRate"]], textposition="outside",
    textfont=dict(color=MUTED),
    customdata=sp["Customers"],
    hovertemplate="%{x}<br>Churn rate %{y:.1f}%<br>%{customdata:,} customers<extra></extra>"))
f_segchurn.update_yaxes(title="Churn rate (%)", range=[0, 100])
f_segchurn = style(f_segchurn, 380, "Churn rate by segment")

t_seg = table(sp.round(2), 240, "Segment profiles (mean values)")

# ================= TAB 4 — REVENUE =================
f_rev = go.Figure(go.Scatter(x=monthly["month"], y=monthly["revenue"], mode="lines+markers",
                             line=dict(color=CAT[0], width=2), marker=dict(size=8),
                             hovertemplate="%{x}<br>£%{y:,.0f}<extra></extra>"))
f_rev.update_yaxes(title="Revenue (GBP)")
f_rev = style(f_rev, 400, "Monthly revenue")
f_rev.update_layout(hovermode="x unified")

f_ord = go.Figure(go.Scatter(x=monthly["month"], y=monthly["orders"], mode="lines+markers",
                             line=dict(color=CAT[1], width=2), marker=dict(size=8),
                             hovertemplate="%{x}<br>%{y:,} orders<extra></extra>"))
f_ord.update_yaxes(title="Orders")
f_ord = style(f_ord, 400, "Monthly order count")
f_ord.update_layout(hovermode="x unified")

tier_order = ["Bronze","Silver","Gold","Platinum"]
pivot = (scores.pivot_table(index="value_tier", columns="risk_band",
                            values="revenue_at_risk", aggfunc="sum")
         .reindex(tier_order).fillna(0))
f_tier = go.Figure()
for bnd in ["Low","Medium","High"]:
    if bnd in pivot.columns:
        f_tier.add_trace(go.Bar(x=pivot.index, y=pivot[bnd], name=bnd + " risk",
                                marker=dict(color=RISK[bnd], line=dict(color=SURF, width=2)),
                                hovertemplate=bnd+" risk · %{x}<br>£%{y:,.0f}<extra></extra>"))
f_tier.update_yaxes(title="Revenue at risk (GBP)")
f_tier.update_layout(barmode="group")
f_tier = style(f_tier, 430, "Revenue at risk by customer value tier")

# ================= TAB 5 — RETENTION LIST =================
top = (scores[(scores.risk_band == "High") & (scores.high_value == 1)]
       .head(20)[["customer_id","segment","monetary","frequency","recency",
                  "churn_probability","revenue_at_risk","recommended_action"]])
top.columns = ["Customer","Segment","Lifetime Value (£)","Orders","Recency (days)",
               "Churn Risk","Revenue at Risk (£)","Action"]
t_top = table(top.round(3), 620, "Top 20 high-value customers at high churn risk")

# ================= ASSEMBLE THE PAGE =================
def div(fig, first=False):
    return fig.to_html(full_html=False, include_plotlyjs=False,
                       config={"displayModeBar": False, "responsive": True})

kpis = [("5,281", "customers scored"),
        (f"£{scores['monetary'].sum()/1e6:.2f}M", "total lifetime value"),
        (f"£{scores['revenue_at_risk'].sum()/1e6:.2f}M", "revenue at risk"),
        (f"{(scores.risk_band=='High').sum():,}", "high-risk customers"),
        (f"{(scores.recommended_action=='Urgent retention call').sum():,}", "urgent retention calls")]
kpi_html = "".join(
    f'<div class="kpi"><div class="kpi-v">{v}</div><div class="kpi-l">{l}</div></div>' for v, l in kpis)

TABS = [("overview", "Overview"), ("models", "Model comparison"),
        ("segments", "Segments"), ("revenue", "Revenue"), ("retention", "Retention list")]
tabbar = "".join(f'<button class="tab{" on" if i==0 else ""}" data-t="{k}">{n}</button>'
                 for i, (k, n) in enumerate(TABS))

html = f"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Customer Analytics Dashboard — DA3</title>
<script>{get_plotlyjs()}</script>
<style>
 :root {{ --surf:{SURF}; --ink:{INK}; --muted:{MUTED}; --grid:{GRID}; --card:#ffffff; }}
 *{{box-sizing:border-box}}
 body{{margin:0;background:#f4f1ea;color:var(--ink);
   font-family:"Segoe UI",Inter,Arial,sans-serif}}
 header{{background:var(--card);padding:26px 32px 18px;border-bottom:1px solid #e3ded4}}
 h1{{margin:0 0 4px;font-size:25px;font-weight:600}}
 .sub{{color:var(--muted);font-size:14px}}
 .kpis{{display:flex;flex-wrap:wrap;gap:14px;padding:20px 32px 6px}}
 .kpi{{background:var(--card);border:1px solid #e6e1d7;border-radius:10px;
   padding:16px 22px;min-width:168px;flex:1}}
 .kpi-v{{font-size:27px;font-weight:600;color:#2A6FB0}}
 .kpi-l{{font-size:12.5px;color:var(--muted);margin-top:3px}}
 .tabs{{display:flex;gap:8px;flex-wrap:wrap;padding:16px 32px 0}}
 .tab{{background:transparent;border:1px solid #ddd6ca;border-radius:999px;
   padding:8px 18px;font-size:14px;color:var(--muted);cursor:pointer}}
 .tab.on{{background:#2A6FB0;border-color:#2A6FB0;color:#fff}}
 .panel{{display:none;padding:18px 32px 44px}} .panel.on{{display:block}}
 .card{{background:var(--card);border:1px solid #e6e1d7;border-radius:10px;
   padding:10px 14px;margin-bottom:18px}}
 .row{{display:flex;gap:18px;flex-wrap:wrap}} .row>.card{{flex:1;min-width:330px}}
 .note{{color:var(--muted);font-size:13px;margin:0 0 14px;line-height:1.55}}
 footer{{color:var(--muted);font-size:12px;padding:0 32px 30px}}
 @media (max-width:760px){{ header,.kpis,.tabs,.panel,footer{{padding-left:16px;padding-right:16px}} }}
</style></head><body>
<header>
  <h1>Customer Analytics Dashboard</h1>
  <div class="sub">Online Retail II · Digital Assignment 3 · Suyash Agrawaal (24BDS1167) · Programming for Data Science</div>
</header>
<div class="kpis">{kpi_html}</div>
<div class="tabs">{tabbar}</div>

<div class="panel on" id="overview">
  <p class="note">Each customer has a churn probability from the tuned Gradient Boosting model (threshold 0.453).
  Revenue at risk is that customer's lifetime spend multiplied by their churn probability.</p>
  <div class="card">{div(f_act)}</div>
  <div class="row"><div class="card">{div(f_risk)}</div><div class="card">{div(f_riskrev)}</div></div>
</div>

<div class="panel" id="models">
  <p class="note">Five churn models compared on the same 1,321-customer test set and the same five cross-validation folds.
  Error bars are bootstrap 95% confidence intervals from 1,000 resamples.</p>
  <div class="card">{div(f_auc)}</div>
  <div class="row"><div class="card">{div(f_box)}</div><div class="card">{div(f_roc)}</div></div>
  <div class="card">{div(t_final)}</div>
  <div class="row"><div class="card">{div(t_pair)}</div><div class="card">{div(t_mcn)}</div></div>
</div>

<div class="panel" id="segments">
  <p class="note">Four K-Means segments on scaled RFM. Bubble size is order frequency; spend is on a log scale
  because the wholesale accounts are orders of magnitude above the rest.</p>
  <div class="card">{div(f_seg)}</div>
  <div class="row"><div class="card">{div(f_segchurn)}</div><div class="card">{div(t_seg)}</div></div>
</div>

<div class="panel" id="revenue">
  <div class="card">{div(f_rev)}</div>
  <div class="card">{div(f_ord)}</div>
  <div class="card">{div(f_tier)}</div>
</div>

<div class="panel" id="retention">
  <p class="note">The actionable output: valuable customers the model places in the high-risk band, ranked by revenue at risk.</p>
  <div class="card">{div(t_top)}</div>
</div>

<footer>Data: Online Retail II (UCI, DOI 10.24432/C5CG6D) · Models trained in Python, data served from SQLite · Generated from DA3_Final_Analysis.ipynb</footer>
<script>
 document.querySelectorAll('.tab').forEach(function(b){{
   b.addEventListener('click', function(){{
     document.querySelectorAll('.tab').forEach(function(x){{x.classList.remove('on');}});
     document.querySelectorAll('.panel').forEach(function(p){{p.classList.remove('on');}});
     b.classList.add('on');
     document.getElementById(b.dataset.t).classList.add('on');
     window.dispatchEvent(new Event('resize'));
   }});
 }});
</script></body></html>"""

with open("DA3_Dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)

import os
print("DA3_Dashboard.html written:", round(os.path.getsize("DA3_Dashboard.html")/1e6, 2), "MB")

DA3_Dashboard.html written: 4.91 MB


In [37]:
from sklearn.metrics import roc_curve

# rebuild probabilities if the runtime was restarted
try:
    probas, yt
except NameError:
    for name, m in models.items():
        m.fit(X_train_s, y_train)
    probas = {name: m.predict_proba(X_test_s)[:, 1] for name, m in models.items()}
    yt = y_test.values

roc_rows = []
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(yt, p)
    step = max(1, len(fpr)//200)
    for f, t in zip(fpr[::step], tpr[::step]):
        roc_rows.append({"Model": name, "FPR": round(float(f), 4), "TPR": round(float(t), 4)})

pd.DataFrame(roc_rows).to_csv("da3_roc_points.csv", index=False)

# confirm every file the dashboard needs is present
import os
need = ["da3_final_comparison.csv","da3_cv_folds.csv","da3_bootstrap_ci.csv",
        "da3_pairwise_tests.csv","da3_mcnemar_matrix.csv","da3_roc_points.csv",
        "da3_customer_scores.csv","da3_segment_profiles.csv","da3_risk_bands.csv",
        "da3_monthly_revenue.csv","customer_features.csv"]
for f in need:
    print(("OK   " if os.path.exists(f) else "MISS "), f)

OK    da3_final_comparison.csv
OK    da3_cv_folds.csv
OK    da3_bootstrap_ci.csv
OK    da3_pairwise_tests.csv
OK    da3_mcnemar_matrix.csv
OK    da3_roc_points.csv
OK    da3_customer_scores.csv
OK    da3_segment_profiles.csv
OK    da3_risk_bands.csv
OK    da3_monthly_revenue.csv
OK    customer_features.csv


In [38]:
import sqlite3, pandas as pd

df     = pd.read_csv("customer_features.csv")
scores = pd.read_csv("da3_customer_scores.csv")

# 1. segment profiles
seg_feats = ["Recency","Frequency","Monetary","AvgOrderValue","AvgQuantity","TotalQuantity"]
seg_profile = df.groupby("Segment")[seg_feats].mean().round(2)
seg_profile["Customers"] = df.groupby("Segment").size()
seg_profile["ChurnRate"] = (df.groupby("Segment")["Churn"].mean()*100).round(1)
seg_profile.reset_index().to_csv("da3_segment_profiles.csv", index=False)

# 2. risk bands
(scores.groupby("risk_band")
       .agg(customers=("customer_id","size"),
            avg_churn_prob=("churn_probability","mean"),
            avg_lifetime_value=("monetary","mean"),
            total_value=("monetary","sum"),
            revenue_at_risk=("revenue_at_risk","sum"))
       .round(2).reset_index()
       .to_csv("da3_risk_bands.csv", index=False))

# 3. monthly revenue from the database
conn = sqlite3.connect("da2_retail_normalized.sqlite")
pd.read_sql("""
SELECT SUBSTR(invoice_date,1,7) AS month,
       COUNT(DISTINCT invoice)  AS orders,
       ROUND(SUM(revenue),2)    AS revenue
FROM transactions GROUP BY month ORDER BY month""", conn).to_csv("da3_monthly_revenue.csv", index=False)
conn.close()

import os
need = ["da3_final_comparison.csv","da3_cv_folds.csv","da3_bootstrap_ci.csv",
        "da3_pairwise_tests.csv","da3_mcnemar_matrix.csv","da3_roc_points.csv",
        "da3_customer_scores.csv","da3_segment_profiles.csv","da3_risk_bands.csv",
        "da3_monthly_revenue.csv","customer_features.csv"]
for f in need:
    print(("OK   " if os.path.exists(f) else "MISS "), f)

OK    da3_final_comparison.csv
OK    da3_cv_folds.csv
OK    da3_bootstrap_ci.csv
OK    da3_pairwise_tests.csv
OK    da3_mcnemar_matrix.csv
OK    da3_roc_points.csv
OK    da3_customer_scores.csv
OK    da3_segment_profiles.csv
OK    da3_risk_bands.csv
OK    da3_monthly_revenue.csv
OK    customer_features.csv


In [39]:
from IPython.display import IFrame, HTML
HTML(open("DA3_Dashboard.html").read()[:0] or "")   # keeps output clean
IFrame(src="DA3_Dashboard.html", width="100%", height=700)

In [40]:
from google.colab import files
files.download("DA3_Dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
for f in ["da3_customer_scores.csv","da3_final_comparison.csv",
          "da3_segment_profiles.csv","da3_monthly_revenue.csv","customer_features.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
   import glob
   for f in sorted(glob.glob("da3_*.csv")): print(f)

da3_bootstrap_ci.csv
da3_confidence_intervals.csv
da3_customer_scores.csv
da3_cv_folds.csv
da3_final_comparison.csv
da3_mcnemar_matrix.csv
da3_monthly_revenue.csv
da3_pairwise_tests.csv
da3_risk_bands.csv
da3_roc_points.csv
da3_segment_profiles.csv


In [43]:
import glob, zipfile
from google.colab import files

csvs = sorted(glob.glob("da3_*.csv"))
print(f"{len(csvs)} CSV files found:")
for f in csvs:
    print("  ", f)

with zipfile.ZipFile("da3_csv_files.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in csvs:
        z.write(f)

files.download("da3_csv_files.zip")

11 CSV files found:
   da3_bootstrap_ci.csv
   da3_confidence_intervals.csv
   da3_customer_scores.csv
   da3_cv_folds.csv
   da3_final_comparison.csv
   da3_mcnemar_matrix.csv
   da3_monthly_revenue.csv
   da3_pairwise_tests.csv
   da3_risk_bands.csv
   da3_roc_points.csv
   da3_segment_profiles.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>